# Example 1 — Supervised Learning: Classification

Student Academic Risk Prediction

**Student guide:** Run cells in order. Read the comments before modifying code. Document each change when adapting the example to a thesis dataset.


## Pipeline
Synthetic dataset creation -> inspection -> preprocessing -> feature/target definition -> train/test split -> naive baseline -> Logistic Regression -> Decision Tree/KNN/Random Forest comparison -> classification metrics -> confusion matrix -> cross-validation -> introductory tuning -> error interpretation.


In [ ]:
# Import common libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay


In [ ]:
# PHASE 1 — CREATE A SYNTHETIC STUDENT-RISK DATASET
# GUIDE: Import the libraries needed for data handling, preprocessing, modeling, and evaluation.

n=300
df=pd.DataFrame({
 "Student_ID":range(1,n+1),
 "Sex":np.random.choice(["Female","Male"],n),
 "Attendance":np.clip(np.random.normal(82,11,n),45,100).round(1),
 "GPA":np.clip(np.random.normal(2.3,.55,n),1,4).round(2),
 "Study_Hours":np.clip(np.random.normal(8,3,n),1,18).round(1),
 "Failed_Subjects":np.random.choice([0,1,2,3,4],n,p=[.48,.25,.14,.08,.05]),
 "Internet_Access":np.random.choice(["Yes","No"],n,p=[.82,.18])
})
score=((df.Attendance<75).astype(int)*2+(df.GPA>2.75).astype(int)*2+
       (df.Study_Hours<6).astype(int)+(df.Failed_Subjects>=2).astype(int)*2+
       (df.Internet_Access=="No").astype(int))
df["Risk_Status"]=np.where(score+np.random.binomial(1,.15,n)>=3,"At Risk","Not At Risk")
display(df.head())
print(df["Risk_Status"].value_counts())


In [ ]:
# PHASE 2 — INSPECTION
# GUIDE: Inspect the dataset before modeling. Check its size, sample rows, missing values, duplicates, and target balance.

print(df.shape)
print(df.dtypes)
print("\nMissing values:\n",df.isna().sum())
print("\nDuplicates:",df.duplicated().sum())
display(df.describe(include="all"))


In [ ]:
# PHASE 3 — DEFINE X AND y
# GUIDE: X contains predictor variables; y contains the known class label.
# GUIDE: Numeric variables are scaled; categorical variables are one-hot encoded.
# GUIDE: Using a Pipeline helps apply the same preprocessing consistently and reduces leakage risk.

# Exclude Student_ID because it is an identifier.
X=df.drop(columns=["Student_ID","Risk_Status"])
y=df["Risk_Status"]

numeric=["Attendance","GPA","Study_Hours","Failed_Subjects"]
categorical=["Sex","Internet_Access"]

# Numerical variables are scaled; categorical variables are one-hot encoded.
prep=ColumnTransformer([
 ("num",StandardScaler(),numeric),
 ("cat",OneHotEncoder(handle_unknown="ignore"),categorical)
])

# PHASE 4 — TRAIN/TEST SPLIT
X_train,X_test,y_train,y_test=train_test_split(
 X,y,test_size=.20,random_state=42,stratify=y)
print(X_train.shape,X_test.shape)


### Understanding the train/test split

The confusion matrix and final test metrics should be based on the **held-out test set**, not the entire dataset. If `test_size=0.20`, approximately 20% of records appear in the confusion matrix; the remaining 80% are used for training.

This separation estimates how the classifier behaves on records it did not use for learning.


In [ ]:
# PHASE 5 — BASELINE AND ALTERNATIVE MODELS
# GUIDE: Dummy is the naive baseline. The other algorithms are learned alternatives.
# GUIDE: Compare Accuracy, Precision, Recall, and F1 rather than relying on Accuracy alone.

models={
 "Dummy":DummyClassifier(strategy="most_frequent"),
 "Logistic Regression":LogisticRegression(max_iter=1000),
 "Decision Tree":DecisionTreeClassifier(max_depth=5,random_state=42),
 "KNN":KNeighborsClassifier(n_neighbors=5),
 "Random Forest":RandomForestClassifier(n_estimators=150,random_state=42)
}
rows=[]
fitted={}
for name,m in models.items():
 pipe=Pipeline([("prep",prep),("model",m)])
 pipe.fit(X_train,y_train)
 p=pipe.predict(X_test)
 fitted[name]=pipe
 rows.append([name,
  accuracy_score(y_test,p),
  precision_score(y_test,p,pos_label="At Risk",zero_division=0),
  recall_score(y_test,p,pos_label="At Risk",zero_division=0),
  f1_score(y_test,p,pos_label="At Risk",zero_division=0)])
results=pd.DataFrame(rows,columns=["Model","Accuracy","Precision","Recall","F1"])
display(results.round(3))


### How to interpret the model-comparison results

Use the metrics according to the research problem:

- **Accuracy** — proportion of all test cases predicted correctly.
- **Precision (At Risk)** — among students predicted *At Risk*, how many are actually At Risk?
- **Recall (At Risk)** — among all truly *At Risk* students, how many did the model identify?
- **F1-score** — balances precision and recall.

For an academic-risk application, **recall deserves particular attention** because low recall means more truly At Risk students are missed. However, the appropriate metric should ultimately follow the research objective and consequences of each error type.

The **Dummy** result is the baseline reference. A learned model should be interpreted relative to that starting point.


In [ ]:
# PHASE 6 — CONFUSION MATRIX AND ERROR ANALYSIS
# GUIDE: The confusion matrix shows which test cases were correctly and incorrectly classified.
# GUIDE: Focus especially on false negatives: actual At Risk students predicted as Not at Risk.

chosen=fitted["Logistic Regression"]
pred=chosen.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test,pred)
plt.title("Academic Risk Classification")
plt.show()
print(classification_report(y_test,pred))

# GUIDE:
# Identify false negatives and false positives.
# Which type of error is more consequential for the intended application?


### How to read the confusion matrix

For binary academic-risk classification, interpret the cells conceptually as:

| Actual condition | Predicted At Risk | Predicted Not at Risk |
|---|---|---|
| **At Risk** | **True Positive (TP):** correctly identified At Risk | **False Negative (FN):** At Risk student was missed |
| **Not at Risk** | **False Positive (FP):** unnecessary At Risk flag | **True Negative (TN):** correctly identified Not at Risk |

The **diagonal cells are correct predictions**; off-diagonal cells are mistakes.

A **False Negative** can be especially important in this example because a student who actually needs support could be classified as Not at Risk. A **False Positive** may lead to unnecessary intervention. Students should explain which error matters more for their own thesis context rather than simply reporting the matrix.


In [ ]:
# PHASE 7 — CROSS-VALIDATION
# GUIDE: Cross-validation repeats evaluation across multiple folds to check whether performance is reasonably stable.

cv=cross_val_score(fitted["Logistic Regression"],X,y,cv=5,scoring="accuracy")
print("CV scores:",np.round(cv,3))
print("Mean:",round(cv.mean(),3),"SD:",round(cv.std(),3))


### How to interpret cross-validation

Cross-validation produces several scores rather than a single train/test result. Examine both the **mean** and the variation across folds. Similar fold scores suggest more stable performance; large differences may indicate sensitivity to the particular data split, limited data, or heterogeneous records.

Cross-validation complements the held-out test set; it does not justify repeatedly tuning decisions against the final test data.


In [ ]:
# PHASE 8 — INTRODUCTORY TUNING
# GUIDE: Test genuinely new student profiles only after formal held-out evaluation.
# GUIDE: New-case predictions demonstrate practical use but do not replace test-set metrics.

tree=Pipeline([("prep",prep),("model",DecisionTreeClassifier(random_state=42))])
search=GridSearchCV(tree,{
 "model__max_depth":[2,3,4,5,8,None],
 "model__min_samples_split":[2,5,10]},cv=5,scoring="accuracy")
search.fit(X_train,y_train)
print("Best parameters:",search.best_params_)
print("CV score:",round(search.best_score_,3))
print("Held-out test score:",round(search.best_estimator_.score(X_test,y_test),3))


## Thesis adaptation
Replace the synthetic data with the thesis dataset. Document the unit of analysis, target definition, feature justification, preprocessing, split strategy, baseline, alternative models, metric choice, error consequences, leakage risks, and next experiment.


## PHASE 9 — Interactive Academic Risk Prediction

This section allows the **user/student to enter a new student profile**. The trained Logistic Regression pipeline then predicts either:

- **At Risk**
- **Not At Risk**

The required inputs exactly match the model features used earlier: **Sex, Attendance, GPA, Study Hours, Failed Subjects, and Internet Access**.

> **Teaching note:** This is a synthetic classroom model. Its prediction demonstrates how a classification model can accept new input and produce a class label. It should **not** be used to make real academic decisions about students without appropriate real-world validation, institutional safeguards, and human review.


In [ ]:
# PHASE 9 — INTERACTIVE USER INPUT AND RISK PREDICTION
# GUIDE: Run this cell after all earlier model-training cells have been executed.
# GUIDE: Enter one new student's information when prompted.
# GUIDE: The values are placed into a one-row DataFrame with the SAME features used during training.
# GUIDE: The already-trained pipeline automatically applies preprocessing before prediction.

print("ACADEMIC RISK PREDICTION")
print("------------------------")

# Ask for categorical information.
# Validation loops prevent unsupported entries.
while True:
    sex = input("Sex (Female/Male): ").strip().title()
    if sex in ["Female", "Male"]:
        break
    print("Please enter Female or Male.")

while True:
    internet = input("Internet Access (Yes/No): ").strip().title()
    if internet in ["Yes", "No"]:
        break
    print("Please enter Yes or No.")

# Ask for numerical information and check reasonable input ranges.
while True:
    try:
        attendance = float(input("Attendance percentage (0-100): "))
        if 0 <= attendance <= 100:
            break
        print("Attendance must be between 0 and 100.")
    except ValueError:
        print("Please enter a numerical value.")

while True:
    try:
        gpa = float(input("GPA (1.00-4.00): "))
        if 1.00 <= gpa <= 4.00:
            break
        print("GPA must be between 1.00 and 4.00.")
    except ValueError:
        print("Please enter a numerical value.")

while True:
    try:
        study_hours = float(input("Study hours per week (0 or higher): "))
        if study_hours >= 0:
            break
        print("Study hours cannot be negative.")
    except ValueError:
        print("Please enter a numerical value.")

while True:
    try:
        failed_subjects = int(input("Number of failed subjects (0 or higher): "))
        if failed_subjects >= 0:
            break
        print("Failed subjects cannot be negative.")
    except ValueError:
        print("Please enter a whole number.")

# IMPORTANT:
# The column names below must exactly match the features used to train the model.
new_student = pd.DataFrame({
    "Sex": [sex],
    "Attendance": [attendance],
    "GPA": [gpa],
    "Study_Hours": [study_hours],
    "Failed_Subjects": [failed_subjects],
    "Internet_Access": [internet]
})

print("\nEntered Student Profile:")
display(new_student)

# 'chosen' is the Logistic Regression pipeline trained earlier.
# Because preprocessing is inside the pipeline, we can directly pass the raw feature values.
risk_prediction = chosen.predict(new_student)[0]

print("\nPREDICTED RISK STATUS:", risk_prediction)

# Logistic Regression also provides class probabilities.
# This is useful for demonstrating how strongly the model favors each class.
if hasattr(chosen.named_steps["model"], "predict_proba"):
    probabilities = chosen.predict_proba(new_student)[0]
    classes = chosen.named_steps["model"].classes_

    probability_table = pd.DataFrame({
        "Risk Status": classes,
        "Predicted Probability": probabilities
    })

    probability_table["Predicted Probability"] = (
        probability_table["Predicted Probability"] * 100
    ).round(2).astype(str) + "%"

    print("\nModel Probability Output:")
    display(probability_table)

print("\nNOTE:")
print("This result is produced by a synthetic teaching model.")
print("It is not a validated academic-risk assessment for real students.")


### How to interpret the interactive result

The **Predicted Risk Status** is the class selected by the trained Logistic Regression model from the information entered by the user.

The probability table provides additional context. For example, if the model produces:

| Risk Status | Predicted Probability |
|---|---:|
| At Risk | 78% |
| Not At Risk | 22% |

the model assigns the entered profile more strongly to **At Risk**. This does **not** mean there is a scientifically established “78% chance” that a real student will experience academic difficulty. In this exercise, the probability is the fitted model's estimated class probability based on the synthetic training data.

### Questions for students

After trying several profiles, observe what happens when you change **one variable at a time**:

1. Keep everything the same and lower Attendance.
2. Restore Attendance and increase Failed Subjects.
3. Change Internet Access from Yes to No.
4. Increase or decrease Study Hours.
5. Change GPA while keeping the other features fixed.

Ask: **Did the predicted class or probability change? Why might the model respond that way?**

This provides a simple introduction to **model sensitivity and interpretation** without treating the prediction as a real-world academic judgment.


## Student Interpretation Guide

After running the notebook, answer briefly:

1. How many total records are in the dataset?
2. How many records were used for training and testing?
3. What was the Dummy baseline performance?
4. Which learned models improved over the baseline?
5. Which model had the highest recall for **At Risk**?
6. What do the false negatives mean in this application?
7. What do the false positives mean?
8. Were the cross-validation results stable across folds?
9. Which model would you investigate further, and **what evidence** supports that choice?
10. What data, feature, fairness, privacy, or model limitations should be investigated before system integration?

### Thesis-writing reminder
Do not report only: *“Model X achieved 90% accuracy.”* Explain the dataset split, baseline, metrics, error types, comparison models, validation results, and practical meaning of the findings.
